# M5.1 · OpenVINO 恢复与公平 2×2 系统对照

**核心问题：** CPU Text Encoder 快了 331–480 倍，系统会快 300 倍吗？从真实记录计算三组编码器比值及 A/C×原始/优化客户端的公平因子实验。C1b 的人工质量接受不覆盖 INT8 数值 FAIL。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.zh-CN.md](../course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.zh-CN.md)。切换语言：[English](./06_openvino_and_fair_ab.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
data = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
c0 = data["arms"]["C0_PyTorch_CPU"]["prompts"]
ov = data["arms"]["C1b_OV_BF16"]["prompts"]
for name in ("p1-zh-scene", "p2-en-scene", "p3-poster-text"):
    ratio = c0[name]["warm_median_s"] / ov[name]["warm_median_s"]
    print(f"{name}: PyTorch CPU {c0[name]['warm_median_s']:.3f}s / OV {ov[name]['warm_median_s']:.3f}s = {ratio:.1f}x")
print("These are encoder-only, positive-prompt comparisons, not E2E images/second.")


In [ ]:
study = ns.fair_factorial()
cells = study["cells"]
for name, v in cells.items():
    print(f"{name:10s} n={v['n']} mean={v['mean_s']:.3f}s => {v['images_per_min']:.3f} img/min")
gain_orig = cells["A_original"]["mean_s"] / cells["C_original"]["mean_s"]
gain_fast = cells["A_fast"]["mean_s"] / cells["C_fast"]["mean_s"]
print(f"Fair architecture gain: original client {gain_orig:.4f}x / optimized client {gain_fast:.4f}x")
assert all(v["n"] == 6 for v in cells.values())
assert 1.0 < gain_fast < 1.05
plot_values = {name: v["mean_s"] for name, v in cells.items()}
plot_ylabel, plot_title = "4-request wall seconds", "Matched 2×2 physical A/B"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
registry = ns.read_json("course/data/verified_claims.json")
claims = {x["id"]: x for x in registry["claims"]}
assert claims["M51-INT8-NEGATIVE-FAIL"]["numerical_verdict"] == "FAIL"
assert claims["M51-OWNER-QUALITY-VERDICT"]["human_quality_verdict"] == "OWNER_ACCEPTED_SCOPE_ONLY"
print("C1b owner acceptance: scoped. INT8 numerical quality gate: FAIL. No status is altered.")


In [ ]:
# Factorial attribution: isolate architecture, client and their interaction.
a0, a1 = cells["A_original"]["mean_s"], cells["A_fast"]["mean_s"]
c0wall, c1wall = cells["C_original"]["mean_s"], cells["C_fast"]["mean_s"]
interaction_s = (a1-a0) - (c1wall-c0wall)
print(f"Monitor gain inside A = {a0/a1:.4f}x; inside C = {c0wall/c1wall:.4f}x.")
print(f"Architecture × monitor interaction = {interaction_s:+.3f}s per 4-request batch.")
print("A-fast vs C-fast is the matched-client architecture contrast.")


## 学生思考 / 自检

先比较 C-fast/A-original，再比较 C-fast/A-fast。为什么前者不能用于架构因果归因？分析节省约17GB显存与CPU worker约23GB RSS的系统成本。

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
